# drfn clean: overnight runs (tokenizer check + seed 22)

Same dataset as the clean grid (esubalewchekol/drfn-clean-pack), so nothing is re-uploaded.

**1. Tokenizer check (seed 11).** Clean-grid scores are 0.7-1.3 chrF below the old-data runs, and runs peak earlier. Two things changed: the cleaned text and the word-boundary tokenizer. `bpe_aux_geezrpa_s11_wstok` trains on the CLEANED text with the OLD whitespace tokenizer. Compare with `bpe_aux_geezrpa_s11` from the clean grid (13.50, new tokenizer) and the old-data run (14.16). Near 14.2 means the tokenizer cost quality; near 13.5 means the cleaning did. Its chrF++ is not meaningful (that tokenizer keeps no word boundaries); character chrF is.

**2. Seed 22 of the seed-11 finding:** analyser labels hurt side heads (-0.67) but help both memory models (gmm +0.34, cmm_spec +0.65), and with analyser labels gmm beats side heads (+0.59).

| run | |
|---|---|
| `bpe_aux` geezRPA / analyser | side heads, both label sources |
| `gmm` geezRPA / analyser | grounded memory, both label sources |
| `cmm_spec` analyser | best translator at seed 11 |

In [ ]:
!pip install -q tokenizers sacrebleu
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
import glob, os, shutil, subprocess, sys, json
def find(pat):
    hits = sorted(glob.glob('/kaggle/input/**/' + pat, recursive=True))
    if not hits: raise RuntimeError('not found under /kaggle/input: ' + pat)
    return hits[0]
pack_src = os.path.dirname(find('run_clean.sh'))
PACK = '/kaggle/working/pack'
if not os.path.exists(PACK): shutil.copytree(pack_src, PACK)
os.chdir(PACK)
print('pack', pack_src)
def stream(cmd, env=None):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        if line.startswith('{"step"') or line.startswith("{'step'"): continue
        print(line, end='')
    return p.wait()

In [ ]:
# 1. tokenizer check: old whitespace BPE, trained on the CLEANED training text
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
sys.path.insert(0, PACK)
import data as D
pairs, _ = D.load_pairs('data/age_train.tsv')
os.makedirs('runs/shared', exist_ok=True)
tok = Tokenizer(BPE(unk_token='[UNK]')); tok.pre_tokenizer = Whitespace()
tok.train_from_iterator([s for p in pairs for s in p],
                        BpeTrainer(vocab_size=16000, special_tokens=['[PAD]', '[BOS]', '[EOS]', '[UNK]']))
tok.save('runs/shared/tokenizer_ws.json')
print('exit', stream([sys.executable, '-u', 'train.py', '--variant', 'bpe_aux', '--seed', '11',
                      '--data-dir', 'data', '--tokenizer', 'runs/shared/tokenizer_ws.json',
                      '--label-cache', 'data/labels_geezrpa.json',
                      '--run-dir', 'runs/bpe_aux_geezrpa_s11_wstok']))

In [ ]:
# 2. seed 22 replication
for LABELS, VARIANTS in (("geezrpa", "bpe_aux gmm"), ("analyser", "bpe_aux gmm cmm_spec")):
    env = dict(os.environ, SEED="22", LABELS=LABELS, VARIANTS=VARIANTS)
    print(LABELS, 'exit', stream(['bash', 'run_clean.sh'], env=env))

In [ ]:
rows = []
for d in sorted(glob.glob('runs/*_s*')):
    lg, fe = os.path.join(d, 'train_log.jsonl'), os.path.join(d, 'final_eval.json')
    if not os.path.exists(lg): continue
    ev = [json.loads(l) for l in open(lg) if '"dev_loss"' in l]
    if not ev: continue
    b = min(ev, key=lambda e: e['dev_loss'])
    f = json.load(open(fe)) if os.path.exists(fe) else {}
    pr = (f.get('probe_root') or {}).get('all_types') or {}
    pp = (f.get('probe_pattern') or {}).get('all_types') or {}
    rows.append((os.path.basename(d), len(ev), b['dev_loss'], b['epoch'],
                 f.get('chrF_char_full_dev'), pr.get('consistency'), pr.get('permutation'),
                 (f.get('probe_root') or {}).get('unseen_slot_accuracy'),
                 pp.get('consistency'), pp.get('permutation')))
fmt = lambda v, s='%.3f': 'n/a' if v is None else s % v
print('%-26s %4s %9s %4s %10s %15s %10s %15s' % ('run', 'ep', 'best loss', '@ep',
      'chrF full', 'root vs chance', 'root unseen', 'pattern vs chance'))
for r in rows:
    print('%-26s %4d %9.4f %4d %10s %15s %10s %15s' % (r[0], r[1], r[2], r[3], fmt(r[4], '%.2f'),
          fmt(r[5]) + '/' + fmt(r[6]), ('%.1f%%' % (100 * r[7])) if r[7] is not None else 'n/a',
          fmt(r[8]) + '/' + fmt(r[9])))
print('\nchrF full = character chrF on the FULL dev set (1,741 sentences), invisible characters removed.')
print('Not comparable with earlier numbers: the data and tokenizer changed. Compare runs within this grid.')
open('/kaggle/working/summary.txt', 'w').write(repr(rows))